In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/annotated_cpu/checkpoints/post_cell_27.pickle

In [ ]:
%%RecordEvent
%%time
### cell 28 ###

import re

# 1. Functions for subsetting, combining, and percentage conversion

def grab_subset_of_data_40(df, question_of_interest):
    """
    Subset columns containing the question text, rename to the suffix after '-' and drop rows
    where all answers are NaN.
    """
    return (
        df
        .filter(like=question_of_interest)
        .rename(columns=lambda col: col.rsplit('-', 1)[-1].lstrip())
        .dropna(how='all')
    )


def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_40(
    question_of_interest,
    include_2017=False
):
    """
    For each year (optionally including 2017), grab the subset for the question, tag with year,
    concatenate, and return both the combined DataFrame and the per-year counts.
    """
    sources = [
        ('2018', responses_df_2018_cell10),
        ('2019', responses_df_2019_cell10),
        ('2020', responses_df_2020),
        ('2021', responses_df_2021),
        ('2022', responses_df_2022_cell10),
    ]
    if include_2017:
        sources.insert(0, ('2017', responses_df_2017))

    dfs = []
    for year, src in sources:
        sub = grab_subset_of_data_40(src, question_of_interest)
        if not sub.empty:
            dfs.append(sub.assign(year=year))

    combined = pd.concat(dfs, ignore_index=True)
    counts = combined.groupby('year', sort=True).count().reset_index()
    return combined, counts


def convert_df_of_counts_to_percentages_40(df, df_counts):
    """
    Given the full concatenated DataFrame df (with a 'year' column) and its per-year counts,
    convert those counts to percentages of respondents per year.
    """
    # Total respondents per year (rows with at least one non-NaN answer)
    totals = df.groupby('year', sort=True).size()

    # Compute percentages
    return (
        df_counts
        .set_index('year')
        .div(totals, axis=0)
        .mul(100)
        .reset_index()
    )


# 2. Standardize question text in 2020 and 2021 column names
pattern_q = re.escape(q_orig)
for df in (responses_df_2020, responses_df_2021):
    df.columns = df.columns.str.replace(pattern_q, q_new, regex=True)


# 3. Bulk rename and replace answers in the 2019 and 2018 DataFrames
for df, mapping in (
    (responses_df_2019_cell10, replacements_2019),
    (responses_df_2018_cell10, replacements_2018),
):
    # Escape keys to match literal substrings
    escaped = {re.escape(old): new for old, new in mapping.items()}
    pattern = '|'.join(escaped.keys())
    # Rename columns (substring replace)
    df.columns = df.columns.str.replace(pattern, lambda m: mapping[m.group()], regex=True)
    # Replace values across the DataFrame in one pass
    df.replace(escaped, regex=True, inplace=True)


# 4. Combine responses, count, convert to percentages, and reshape for plotting
notebooks_df_combined, notebooks_df_combined_counts = (
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_40(
        question_of_interest_cell40
    )
)

notebooks_df_combined_percentages = convert_df_of_counts_to_percentages_40(
    notebooks_df_combined,
    notebooks_df_combined_counts
)

# Melt directly with var_name to avoid a separate rename
df_cell40 = pd.melt(
    notebooks_df_combined_percentages,
    id_vars='year',
    value_vars=['None', 'Kaggle Notebooks', 'Colab Notebooks'],
    var_name='',
    value_name='value'
)

df_cell40.info()

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_28_try_3.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_28_try_3.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[28], f)


In [ ]:
opt_output = Out.get(4)